In [0]:
df = spark.read.format('parquet').load('/Volumes/dev/anttyjosh3831/dataset/files/orders_first.parquet')
df.display()

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import current_timestamp
from utilities.utils import bronze_schema_expectations, make_to_struct

In [0]:
enforced_schema = make_to_struct(bronze_schema_expectations)

In [0]:
@dp.table(
    name = 'bronze_orders',
    partition_columns = ['order_date'],
    properties = {
        'pipelines.autoOptimize.managed': 'true',
        'delta.autoOptimize.optimizeWrite': 'true',
        'delta.autoOptimize.autoCompact': 'true'
    },
    comment = 'dev'
)

@dp.expect_or_drop('valid_order_id', 'order_id IS NOT NULL')

def orders():
    df = spark.readStream \
        .format('cloudFiles') \
        .option('cloudFiles.format', 'parquet') \
        .option('cloudFiles.schemaLocation', '/Volumes/dev/anttyjosh3831/dataset/orders_schema/') \
        .option('maxFilesPerTrigger', 50) \
        .schema(enforced_schema) \
        .load('/Volumes/dev/anttyjosh3831/dataset/orders/')

    df.withColumn('ingestion_tstmp', current_timestamp())
    return df

In [0]:
@dp.table(
    name='bronze_customers'
)

In [0]:
df = spark.read.format('parquet').load('/Volumes/dev/anttyjosh3831/dataset/customers/')
df.display()
